In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [ ]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [14]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gemma4",
    model_provider="openai",
    api_key="dummy",
    base_url="http://localhost:8080/v1",
)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=prompt
)

In [10]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [11]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='8c182aca-e032-4895-b265-f1782e802490'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 208, 'total_tokens': 230, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gemma-4-E4B-it-UD-Q4_K_XL.gguf', 'system_fingerprint': 'b9670-02810c7aa', 'id': 'chatcmpl-AS478bIPNZa5FfmXOaMHeRACsRK9hdnf', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a08086-c77c-7ba3-9efe-1de19daec642-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': 'Q1jocEMtXDKXlOMZeyk5y0AyAvKc97zY', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 208, 'output_tokens': 22, 'total_toke

## Online MCP

In [15]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [16]:
agent = create_agent(
    model=model,
    tools=tools,
)

In [ ]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it in india?', additional_kwargs={}, response_metadata={}, id='f2105ce7-b999-42d8-86ad-eba6b4516d0a'),
              AIMessage(content="What timezone are you in? If you don't specify, I will use 'America/New_York'.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 299, 'total_tokens': 323, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 283}}, 'model_provider': 'openai', 'model_name': 'gemma-4-E4B-it-UD-Q4_K_XL.gguf', 'system_fingerprint': 'b9670-02810c7aa', 'id': 'chatcmpl-UnaC9ZN0zDTTZJL4tuntzo4GD9G3FuZZ', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0808b-1ac7-7362-98a1-9e882d704d5d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 299, 'output_tokens': 24, 'total_tokens': 323, 'input_token_details': {'cache_read': 283}, 'output_token_details': {}})]}
